# SaudiIntent AI: Task 4, MARBERTv2 fine-tuning (Google Colab)

Run the cells from top to bottom. Before you start: **Runtime > Change runtime type > T4 GPU**.

What this notebook does:
1. Gets the repo and installs the libraries.
2. Runs the hyper-parameter search (3 learning rates x 2 epoch counts = 6 runs), chosen on the **validation** set.
3. Shows the results.
4. Writes the predictions on the Saudi test set for Task 5 (evaluation). The test set is only *predicted*, never used to choose settings.
5. Saves the model (Google Drive and/or the Hugging Face Hub).

## 1. Check the GPU

In [ ]:
!nvidia-smi

## 2. Get the repo
**Option A (after the repo is on GitHub):** replace `<your-username>` and run.
**Option B:** upload a zip of the repo with the Files panel on the left, then use the commented lines instead.

In [ ]:
!git clone https://github.com/<your-username>/saudiintent-ai.git
%cd saudiintent-ai

# Option B (zip upload):
# !unzip -q saudiintent-ai.zip
# %cd saudiintent-ai

## 3. Install the libraries (Colab already has PyTorch)

In [ ]:
!pip install -q -r requirements-marbert.txt

## 4. Check the data (takes seconds)

In [ ]:
!python -m pytest -q tests/

## 5. Train: the full search
Settings are in `src/marbert/config.yaml`. Expect roughly a few minutes per run on a T4.
For a quick first try, run one setting instead: `!python src/marbert/train.py --lr 2e-5 --epochs 3`

In [ ]:
!python src/marbert/train.py

## 6. Results

In [ ]:
import json, pandas as pd
display(pd.read_csv("results/marbert/search_results.csv"))
print(json.dumps(json.load(open("results/marbert/best_params.json"))["best"], indent=2))
print(open("results/marbert/validation_report_best.txt").read())

## 7. Try your own messages

In [ ]:
import sys; sys.path.insert(0, "src")
from marbert.predict import IntentPredictor
predictor = IntentPredictor("models/marbert/best")
for msg in ["وين وصل طلبي؟", "ابي اكلم موظف", "البطاقة ما قبلت", "abi a cancel el order"]:
    r = predictor.predict(msg)
    print(f"{msg}  ->  {r['intent']}  ({r['confidence']:.2f})")

## 8. Predictions on the Saudi test set (for Task 5)
Run this **once**, after the search is finished. Do not go back and change settings because of these results:
that would make the final test unfair.

In [ ]:
!python src/marbert/predict.py --input data/saudi_test/saudi_test_FINAL.csv --output results/marbert/predictions_saudi_test.csv

## 9. Save your work
Colab deletes everything when the session ends.
- **Results** (`results/marbert/`): small; download them and commit them to your `task4-marbert` branch.
- **Model** (`models/marbert/best/`, about 650 MB): too big for GitHub. Save it to Google Drive and/or the Hugging Face Hub.

In [ ]:
# Download the results folder as a zip
!zip -qr marbert_results.zip results/marbert
from google.colab import files
files.download("marbert_results.zip")

In [ ]:
# Save the model to Google Drive
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/saudiintent && cp -r models/marbert/best /content/drive/MyDrive/saudiintent/marbert-best

In [ ]:
# Optional: upload the model to the Hugging Face Hub.
# Put your token in Colab Secrets (key icon on the left) with the name HF_TOKEN. Never paste it in the notebook.
# from google.colab import userdata
# from huggingface_hub import login
# login(token=userdata.get("HF_TOKEN"))
# from transformers import AutoModelForSequenceClassification, AutoTokenizer
# repo_id = "<your-hf-username>/saudiintent-marbertv2"
# AutoModelForSequenceClassification.from_pretrained("models/marbert/best").push_to_hub(repo_id, private=True)
# AutoTokenizer.from_pretrained("models/marbert/best").push_to_hub(repo_id, private=True)